# Stage 2: GDELT News Collection

Collect raw GDELT tone timelines and article metadata, then cache to parquet/json.

In [1]:
import json
import hashlib
import time
from pathlib import Path
import pandas as pd
import requests
import yfinance as yf
from tqdm import tqdm

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
DATA_DIR = ROOT / 'data' / 'intermediate'
CACHE_DIR = ROOT / 'data' / 'cache' / 'gdelt'
CACHE_DIR.mkdir(parents=True, exist_ok=True)

tickers = pd.read_csv(DATA_DIR / 'sp500_tickers.csv')['ticker'].tolist()
close = pd.read_parquet(DATA_DIR / 'close_prices.parquet')
fetch_start = pd.to_datetime(close.index.min())
fetch_end = pd.to_datetime(close.index.max())
gdelt_start = max(pd.Timestamp('2017-02-01'), fetch_start)

print('Tickers:', len(tickers), 'GDELT window:', gdelt_start.date(), '->', fetch_end.date())

Tickers: 503 GDELT window: 2017-02-01 -> 2026-03-06


/Users/5940365/Desktop/CS229Project/cs229project/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
GDELT_BASE = 'https://api.gdeltproject.org/api/v2/doc/doc'

def clean_name(name):
    if not name:
        return None
    for suf in [', Inc.', ' Inc.', ' Inc', ', Corp.', ' Corp.', ' Corp',
                ' Corporation', ' Company', ' Co.', ', Ltd.', ' Ltd.',
                ' Ltd', ' PLC', ' plc', ' N.V.', ' S.A.', ' AG', ' SE',
                ' Holdings', ' Group', ' Enterprises', ', L.P.']:
        name = name.replace(suf, '')
    return name.strip()

def cache_path(mode, params):
    key = hashlib.md5(json.dumps(params, sort_keys=True).encode()).hexdigest()
    return CACHE_DIR / f'{mode}_{key}.json'

def query_gdelt(query, mode, start_dt, end_dt, maxrecords=250):
    params = {
        'query': query,
        'mode': mode,
        'format': 'json',
        'sourcelang': 'english',
        'startdatetime': start_dt,
        'enddatetime': end_dt
    }
    if mode == 'artlist':
        params['maxrecords'] = maxrecords

    cp = cache_path(mode, params)
    if cp.exists():
        return json.loads(cp.read_text())

    try:
        r = requests.get(GDELT_BASE, params=params, timeout=60)
        r.raise_for_status()
        data = r.json()
        cp.write_text(json.dumps(data))
        return data
    except Exception:
        return None

def parse_timeline(data):
    if not data or 'timeline' not in data:
        return pd.Series(dtype=float)
    rows = []
    for block in data['timeline']:
        series = block.get('data', block.get('series', []))
        if isinstance(series, str):
            for line in series.strip().split('\n'):
                parts = line.rsplit(',', 1)
                if len(parts) == 2:
                    try:
                        rows.append((pd.to_datetime(parts[0].strip()), float(parts[1])))
                    except Exception:
                        pass
        elif isinstance(series, list):
            for pt in series:
                try:
                    rows.append((pd.to_datetime(pt.get('date', pt.get('bin', ''))), float(pt.get('value', pt.get('count', 0)))))
                except Exception:
                    pass
    if not rows:
        return pd.Series(dtype=float)
    s = pd.DataFrame(rows, columns=['date', 'value']).set_index('date')['value']
    return s.groupby(level=0).mean().sort_index()

def parse_artlist(data):
    if not data or 'articles' not in data:
        return []
    out = []
    for a in data['articles']:
        title = a.get('title', '').strip()
        seen = a.get('seendate', '')
        if title and seen:
            try:
                out.append({
                    'date': pd.to_datetime(seen).normalize(),
                    'title': title,
                    'domain': a.get('domain', ''),
                    'url': a.get('url', '')
                })
            except Exception:
                pass
    return out

In [3]:
name_cache_file = DATA_DIR / 'ticker_to_name.json'
if name_cache_file.exists():
    ticker_names = json.loads(name_cache_file.read_text())
else:
    ticker_names = {}
    for t in tqdm(tickers, desc='Resolving company names'):
        try:
            info = yf.Ticker(t).get_info()
            raw = info.get('shortName') or info.get('longName') or t
            ticker_names[t] = clean_name(raw)
        except Exception:
            ticker_names[t] = t
        time.sleep(0.1)
    name_cache_file.write_text(json.dumps(ticker_names))

print('Resolved names:', len(ticker_names))

Resolved names: 503


In [4]:
from concurrent.futures import ThreadPoolExecutor, as_completed

start_fmt = gdelt_start.strftime('%Y%m%d%H%M%S')
end_fmt = fetch_end.strftime('%Y%m%d%H%M%S')
yearly_starts = pd.date_range(gdelt_start, fetch_end, freq='YS')

def fetch_ticker(t):
    """Fetch tone timeline + yearly article lists for one ticker."""
    name = ticker_names.get(t, t)
    query = f'"{name}"'

    tone_data = query_gdelt(query, 'timelinetone', start_fmt, end_fmt)
    tone_series = parse_timeline(tone_data)

    arts = []
    for ys in yearly_starts:
        ye = min(ys + pd.DateOffset(years=1) - pd.Timedelta(days=1), fetch_end)
        raw = query_gdelt(query, 'artlist',
                          ys.strftime('%Y%m%d%H%M%S'),
                          ye.strftime('%Y%m%d%H%M%S'),
                          maxrecords=250)
        for a in parse_artlist(raw):
            arts.append({
                'ticker': t,
                'date': a['date'],
                'title': a['title'],
                'domain': a['domain'],
                'url': a['url']
            })
    return t, tone_series, arts

tone_dict = {}
article_rows = []
WORKERS = 12  # parallel HTTP threads

with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    futures = {pool.submit(fetch_ticker, t): t for t in tickers}
    for fut in tqdm(as_completed(futures), total=len(futures), desc='GDELT collection'):
        t, tone_series, arts = fut.result()
        if not tone_series.empty:
            tone_dict[t] = tone_series
        article_rows.extend(arts)

tone_wide = pd.DataFrame(tone_dict)
tone_wide.index = pd.to_datetime(tone_wide.index)
articles_raw = pd.DataFrame(article_rows)
if not articles_raw.empty:
    articles_raw['date'] = pd.to_datetime(articles_raw['date'])

tone_wide.to_parquet(DATA_DIR / 'gdelt_tone_wide.parquet')
articles_raw.to_parquet(DATA_DIR / 'gdelt_articles_raw.parquet')
print('Saved gdelt_tone_wide.parquet and gdelt_articles_raw.parquet')
print('Tone shape:', tone_wide.shape, 'Articles:', len(articles_raw))

GDELT collection: 100%|██████████| 503/503 [04:50<00:00,  1.73it/s]

Saved gdelt_tone_wide.parquet and gdelt_articles_raw.parquet
Tone shape: (3300, 33) Articles: 53075
